In [1]:
using Falcons
include("../src/EBC.jl")
include("../../BeamToyModel/src/BeamToyModel.jl")
using Base.Threads
using Statistics
using NPZ

In [27]:
"""
Return the inclusive pixel index range (in RING ordering) for a given ring.

Arguments
- ring::Int : HEALPix ring index, 1 <= ring <= 4nside-1
- nside::Int

Returns
- (start, stop)::Tuple{Int,Int}
"""
function ring_pixel_range(ring::Int, nside::Int)
    nrings = 4nside - 1
    if ring < 1 || ring > nrings
        throw(ArgumentError("ring must be in 1:$nrings for nside=$nside, got $ring"))
    end

    nphi = pixels_in_ring(ring, nside)
    start = first_pixel_in_ring(ring, nside)
    stop  = start + nphi - 1
    return start, stop
end

"""
Number of pixels in a given ring (RING ordering).
"""
function pixels_in_ring(ring::Int, nside::Int)
    if ring <= nside
        return 4ring
    elseif ring <= 3nside
        return 4nside
    else
        return 4(4nside - ring)
    end
end

"""
First pixel index (1-based, RING ordering) of a given ring.
"""
function first_pixel_in_ring(ring::Int, nside::Int)
    if ring <= nside
        # sum_{k=1}^{ring-1} 4k + 1
        return 2ring*(ring - 1) + 1
    elseif ring <= 3nside
        # north cap pixels + equatorial offset + 1
        northcap = 2nside*(nside - 1)
        return northcap + (ring - nside - 1) * 4nside + 1
    else
        # use south-side symmetry
        nrings = 4nside - 1
        r′ = nrings - ring + 1   # mirrored ring counted from south pole
        npix = 12nside^2
        return npix - 2r′*(r′ + 1) + 1
    end
end

first_pixel_in_ring

In [2]:
ring_pixel_range(125, 32)

(12265, 12276)

In [42]:
pixels_in_ring(125, 32)

12

In [43]:
first_pixel_in_ring(125, 32)

12265

In [40]:
unique_theta_num(127, 32)

(12285, 12288)

In [34]:
npix = nside2npix(32)

12288

In [45]:
function unique_theta_val2(nside::Int)
    nrings = 4nside - 1
    θ = Vector{Float64}(undef, nrings)

    for r in 1:nrings
        z = if r <= nside
            1 - r^2 / (3nside^2)
        elseif r <= 3nside
            (4/3) - (2r)/(3nside)
        else
            rr = 4nside - r
            -1 + rr^2 / (3nside^2)
        end
        θ[r] = acos(z)
    end

    return θ
end

unique_theta_val2 (generic function with 1 method)

In [47]:
test = unique_theta_val2(32)
test2 = unique_theta_val(32)

127-element Vector{Float64}:
 0.02551621035741883
 0.05103657515266638
 0.07656525491520565
 0.10210642238260403
 0.12766426866687355
 0.1532430094961162
 0.17884689155751526
 0.20448019896853498
 0.23014725990424112
 0.25585245340993734
 0.28160021642986005
 0.3073950510845034
 0.33324153223126673
 ⋮
 2.8341976025052897
 2.8599924371599332
 2.885740200179856
 2.911445393685552
 2.9371124546212584
 2.962745762032278
 2.988349644093677
 3.0139283849229197
 3.039486231207189
 3.0650273986745877
 3.090556078437127
 3.1160764432323744

In [51]:
maximum(abs.(test2 .- test))

1.4502288259166107e-15

In [10]:
ss = gen_ScanningStrategy(nside = 32,coord="G")
show_ss(ss)

nside                    : 32 
duration [sec]           : 31536000.0 
sampling rate [Hz]       : 1.0 
alpha [deg]              : 45.0 
beta [deg]               : 50.0 
gamma [deg]              : 0.0 
prec. period [min]       : 192.348
↳ prec. rate [rpm]       : 0.005199
spin period [min]        : 20.000
↳ spin rate [rpm]        : 0.050000
HWP rot. rate[rpm]       : 0.000000 
start angle              : 0.000000 
coordinate system        : G 
FPU                     
↳ Det. 1  name | boresight              : (x,y,z,w) = [0.000, 0.000, 1.000, 0.000] 


In [15]:
nptg = 31536000
pointings = zeros(nptg,3)

pointings[:,1], pointings[:,2], pointings[:,3], time_array = get_pointings(ss, 0, nptg)
;

In [ ]:
ss.coord

ScanningStrategy{Float64, Int64, String}(128, 31536000, 1.0, 45.0, 50.0, 0.0, 0.005198910308399359, 0.05, 0.0, "equator", 0.0, "E", [[0.0, 0.0, 1.0, 0.0]], ["boresight"], 0×0 DataFrame)

In [26]:
nside = 512
12*nside^2/(4nside)

1536.0

In [3]:
nside_in = 32
res = Resolution(nside_in)
lmax_in = 3nside_in-1
fwhm = deg2rad(10.0)
blm_harmonic = BeamToyModel.gaussian_harmonic(lmax_in, fwhm, mmax = 2)
;